# Lezione 5 — Da modellazione a valutazione: costruisci e valida un modello

**Durata stimata: 120 minuti**

## Obiettivi di apprendimento

- Distinguere modelli descrittivi e predittivi, e il ruolo dell'insieme di addestramento.
- Addestrare un albero decisionale con scikit-learn.
- Leggere una matrice di confusione e calcolare l'accuracy per classe.
- Capire cosa misura la curva ROC e come aiuta a scegliere il modello migliore.

Questo notebook è autocontenuto: non devi aver eseguito prima i notebook delle lezioni 3 e 4. Iniziamo ricaricando e pulendo il dataset di ricette da zero, esattamente come è rimasto alla fine della lezione 4, e poi addestriamo e valutiamo un albero decisionale reale con scikit-learn.

## 1. Ripasso rapido: caricare e pulire il dataset di ricette

Prima di modellare servono dati puliti. Questo è un riepilogo del lavoro fatto nelle lezioni 3 e 4: caricare il CSV, mettere in minuscolo e normalizzare i nomi di cucina, togliere cucine con pochissime ricette, e convertire "Yes"/"No" in 1/0.

In [ ]:
import pandas as pd
pd.set_option("display.max_columns", None)
import numpy as np
import re
import random

# Scarica il CSV direttamente dal server di IBM e lo carica come DataFrame
recipes = pd.read_csv(
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBMDeveloperSkillsNetwork-DS0103EN-SkillsNetwork/labs/Module%202/recipes.csv"
)

# La prima colonna porta il nome della cucina, ma senza intestazione chiara: la rinominiamo
column_names = recipes.columns.values
column_names[0] = "cuisine"
recipes.columns = column_names

# Normalizza in minuscolo affinché "Mexican" e "mexican" non contino come cucine distinte
recipes["cuisine"] = recipes["cuisine"].str.lower()

# Corregge nomi di cucina incoerenti (il dataset originale porta diverse varianti)
recipes.loc[recipes["cuisine"] == "korea", "cuisine"] = "korean"
recipes.loc[recipes["cuisine"] == "japan", "cuisine"] = "japanese"
recipes.loc[recipes["cuisine"] == "china", "cuisine"] = "chinese"
recipes.loc[recipes["cuisine"] == "thailand", "cuisine"] = "thai"
recipes.loc[recipes["cuisine"] == "india", "cuisine"] = "indian"

# Elimina cucine con meno di 50 ricette: non ci sono abbastanza esempi per imparare un pattern affidabile
recipes_counts = recipes["cuisine"].value_counts()
cuisines_indices = recipes_counts > 50
cuisines_to_keep = list(np.array(recipes_counts.index.values)[np.array(cuisines_indices)])
recipes = recipes.loc[recipes["cuisine"].isin(cuisines_to_keep)]

# Converte le colonne degli ingredienti da "Yes"/"No" a 1/0, formato che sklearn può elaborare
recipes = recipes.replace(to_replace="Yes", value=1)
recipes = recipes.replace(to_replace="No", value=0)

print("Dataset limpio y listo:", recipes.shape)

## 2. Modellazione: descrittivo vs. predittivo

Prima di addestrare qualsiasi cosa, ricorda la distinzione del capitolo:

- Un **modello descrittivo** esamina relazioni tra variabili per fare inferenze ("se questo, allora probabilmente quello").
- Un **modello predittivo** produce un risultato di tipo sì/no (o, nel nostro caso, "di quale cucina è questa ricetta?").

Costruiremo un modello **predittivo**: dato un insieme di ingredienti (1/0), prevedere la cucina di origine. Per addestrarlo ci serve un **insieme di addestramento**: ricette di cui già conosciamo la cucina reale, affinché l'algoritmo impari il pattern che le collega ai loro ingredienti.

### Caso CHF (promemoria concettuale)

Nel caso dell'assicurazione sanitaria, il primo albero decisionale per prevedere il ricovero ripetuto di pazienti con insufficienza cardiaca congestizia (CHF) ha avuto un 85% di accuracy globale, ma azzeccava solo il 45% dei ricoveri ripetuti reali ("sì"). Regolando il **costo relativo di classificazione errata** (il peso che l'algoritmo dà a sbagliare sul "sì" rispetto al "no"), il team ha provato tre configurazioni:

| Modello | Costo relativo sì:no | Accuracy su "sì" | Accuracy su "no" | Accuracy globale |
|---|---|---|---|---|
| 1 | 1:1 (default) | 45% | — | 85% |
| 2 | 9:1 | 97% | molto bassa | 49% |
| 3 | 4:1 | 68% | 85% | 81% |

Il modello 3 è stato il miglior equilibrio. Vedrai questa stessa logica di valutazione (accuracy per classe, non solo globale) applicata al dataset di ricette più sotto.

## 3. Addestrare un albero decisionale con scikit-learn

Lavoreremo con un sottoinsieme di 5 cucine: coreana, giapponese, cinese, thailandese e indiana. `DecisionTreeClassifier` è la classe di scikit-learn che costruisce alberi decisionali per la classificazione.

In [ ]:
from sklearn import tree

# Filtra il dataset pulito per tenere solo queste 5 cucine
asian_indian_recipes = recipes[recipes.cuisine.isin(
    ["korean", "japanese", "chinese", "thai", "indian"]
)]

# La variabile da prevedere: la cucina di ogni ricetta
cuisines = asian_indian_recipes["cuisine"]

# Le variabili di ingresso: tutte le colonne di ingredienti
# (iloc[:, 1:] scarta la prima colonna, che è "cuisine")
ingredients = asian_indian_recipes.iloc[:, 1:]

# Crea l'albero decisionale, limitando la sua profondità a 3 livelli affinché sia leggibile
bamboo_tree = tree.DecisionTreeClassifier(max_depth=3)

# .fit() addestra il modello: gli mostra ingredienti (ingresso) + cucina reale (uscita)
# di ogni ricetta, e l'algoritmo impara quali domande separano meglio le cucine
bamboo_tree.fit(ingredients, cuisines)

print("Decision tree model saved to bamboo_tree!")

### Visualizzare l'albero: da graphviz a plot_tree

Il notebook originale di IBM visualizza l'albero con la libreria `graphviz`, installata via `!conda install python-graphviz --yes`. **Questo non funziona su Google Colab, perché Colab non ha conda.** Per questo, in questa lezione adattiamo il notebook originale di IBM, che usava graphviz via conda, con `plot_tree` di scikit-learn, affinché funzioni su Colab senza installare nulla.

In [ ]:
import matplotlib.pyplot as plt

# plot_tree disegna l'albero direttamente con matplotlib: non serve graphviz
plt.figure(figsize=(20, 10))
tree.plot_tree(
    bamboo_tree,
    feature_names=list(ingredients.columns.values),
    class_names=sorted(cuisines.unique()),
    filled=True,
    fontsize=8,
)
plt.show()

Leggendo l'albero puoi estrarre regole concrete, ad esempio:

- Se una ricetta contiene **cumino (cumin)** e **pesce (fish)**, ma NON **yogurt (yoghurt)**, il modello la classifica come molto probabilmente **thailandese**.
- Se contiene **cumino** ma NON **pesce** e NON **salsa di soia**, il modello la classifica come molto probabilmente **indiana**.

Questo è il grande vantaggio di un albero decisionale rispetto ad altri modelli: oltre a prevedere, ti lascia "leggere" il ragionamento dietro ogni previsione.

## 4. Valutazione: dividere in train/test

Addestrare e valutare con gli stessi dati sarebbe barare: il modello potrebbe memorizzare invece di imparare un pattern generale. Per questo separiamo un **insieme di test**, che il modello non vede mai durante l'addestramento, per verificare se generalizza bene.

In [ ]:
bamboo = recipes[recipes.cuisine.isin(
    ["korean", "japanese", "chinese", "thai", "indian"]
)]

print(bamboo["cuisine"].value_counts())

In [ ]:
sample_n = 30
random.seed(1234)  # fissa il seme casuale affinché il risultato sia riproducibile

# Prende un campione di 30 ricette per ogni cucina per l'insieme di test
bamboo_test = bamboo.groupby("cuisine", group_keys=False).apply(lambda x: x.sample(sample_n))

bamboo_test_ingredients = bamboo_test.iloc[:, 1:]
bamboo_test_cuisines = bamboo_test["cuisine"]

# L'insieme di addestramento è tutto ciò che NON è finito nell'insieme di test
bamboo_test_index = bamboo.index.isin(bamboo_test.index)
bamboo_train = bamboo[~bamboo_test_index]

bamboo_train_ingredients = bamboo_train.iloc[:, 1:]
bamboo_train_cuisines = bamboo_train["cuisine"]

# Addestra un nuovo albero, questa volta solo con l'insieme di addestramento
bamboo_train_tree = tree.DecisionTreeClassifier(max_depth=15)
bamboo_train_tree.fit(bamboo_train_ingredients, bamboo_train_cuisines)

print("Decision tree model saved to bamboo_train_tree!")

In [ ]:
# .predict() usa il modello già addestrato per prevedere la cucina
# di ricette che il modello NON ha mai visto durante l'addestramento
bamboo_pred_cuisines = bamboo_train_tree.predict(bamboo_test_ingredients)

bamboo_pred_cuisines[:10]

## 5. La matrice di confusione

La **matrice di confusione** è lo strumento centrale per leggere quanto bene ha previsto il modello: le **righe rappresentano la cucina reale** e le **colonne rappresentano la cucina prevista dal modello**. Se il modello fosse perfetto, tutto il "peso" starebbe sulla diagonale (dove riga e colonna coincidono). Convertiamo ogni riga in percentuale, così ogni riga dovrebbe sommare 100%.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix
import itertools

test_cuisines = np.unique(bamboo_test_cuisines)
bamboo_confusion_matrix = confusion_matrix(bamboo_test_cuisines, bamboo_pred_cuisines, labels=test_cuisines)
title = "Bamboo Confusion Matrix"
cmap = plt.cm.Blues

plt.figure(figsize=(8, 6))
bamboo_confusion_matrix = (
    bamboo_confusion_matrix.astype("float") / bamboo_confusion_matrix.sum(axis=1)[:, np.newaxis]
) * 100

plt.imshow(bamboo_confusion_matrix, interpolation="nearest", cmap=cmap)
plt.title(title)
plt.colorbar()
tick_marks = np.arange(len(test_cuisines))
plt.xticks(tick_marks, test_cuisines)
plt.yticks(tick_marks, test_cuisines)

fmt = ".2f"
thresh = bamboo_confusion_matrix.max() / 2.0
for i, j in itertools.product(range(bamboo_confusion_matrix.shape[0]), range(bamboo_confusion_matrix.shape[1])):
    plt.text(
        j, i, format(bamboo_confusion_matrix[i, j], fmt),
        horizontalalignment="center",
        color="white" if bamboo_confusion_matrix[i, j] > thresh else "black",
    )

plt.tight_layout()
plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.show()

Con il seme `1234`, un'esecuzione tipica mostra che circa il **60% delle ricette cinesi** si classifica correttamente (un 37% si confonde con coreana, un 3% con indiana), e circa il **77% delle ricette indiane** si classifica correttamente. Poiché gli alberi decisionali si costruiscono con campionamento casuale, i tuoi numeri possono variare leggermente ogni volta che esegui il codice, ma le prestazioni generali dovrebbero essere comparabili.

Questo è esattamente lo stesso tipo di lettura fatta con la tabella del caso CHF: non basta un numero globale di accuracy, bisogna guardare classe per classe per sapere dove il modello azzecca e dove si confonde.

In [ ]:
# Accuracy globale: proporzione di previsioni corrette sul totale delle ricette di test
accuracy_global = accuracy_score(bamboo_test_cuisines, bamboo_pred_cuisines)
print(f"Accuracy global del modelo: {accuracy_global:.2%}")

## 6. La curva ROC: come si sceglie il modello migliore

La curva [ROC](../manual/01_glosario_IT.md) (*receiver operating characteristic*) è nata come strumento radar nella Seconda Guerra Mondiale, per rilevare aerei nemici. Oggi è standard nel machine learning per confrontare modelli di classificazione binaria.

Confronta, per diversi valori di un criterio di discriminazione (nel caso CHF, il costo relativo di classificazione errata), il **tasso di veri positivi** (sensibilità) contro il **tasso di falsi positivi**. Il modello ottimale è quello che ottiene la **massima separazione** tra la sua curva e la linea base di un modello che indovina a caso.

Nel caso CHF, dei 4 modelli valutati con diversi costi relativi, il **modello 3 (costo 4:1)** è stato quello che ha dato la massima separazione rispetto alla linea base — per questo è stato il modello scelto, non quello con l'accuracy globale più alta (modello 1) né quello che rilevava meglio i "sì" (modello 2).

Questo notebook non costruisce una curva ROC con il dataset di ricette (è un problema multiclasse con 5 cucine, e la curva ROC classica è pensata per la classificazione binaria), ma la logica è la stessa appena vista con la matrice di confusione: non fermarti mai a una sola metrica, confronta il comportamento del modello in tutte le classi e scegli quello che meglio bilancia i diversi tipi di errore.

## Riepilogo

- Hai addestrato un albero decisionale (`DecisionTreeClassifier`) con `.fit()` su ingredienti di ricette per prevedere la loro cucina.
- Hai visualizzato l'albero con `plot_tree` (invece di graphviz, che non funziona su Colab) e hai letto regole concrete di classificazione.
- Hai diviso i dati in addestramento e test, hai previsto con `.predict()`, e hai letto una matrice di confusione reale (righe = reale, colonne = previsto).
- Hai collegato questa valutazione al caso CHF: un'accuracy globale alta non garantisce un buon modello se fallisce nella classe che più conta, e la curva ROC aiuta a scegliere il modello con miglior equilibrio.

Continua con il notebook di pratica di questa lezione per addestrare il tuo albero con un sottoinsieme diverso di cucine.